<a href="https://colab.research.google.com/github/Maddox159-crypto/ESAA_assignment/blob/main/%EB%88%84%EB%A9%94%EB%9D%BC%EC%9D%B4%EF%BC%91.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pyarrow.parquet as pq
import numpy as np
from sklearn.decomposition import IncrementalPCA
import gc

train_path = '/content/drive/MyDrive/ESAA/numerai_r1300__v5_2_train.parquet'

# feature 컬럼 이름만 먼저 가볍게 확인
pf = pq.ParquetFile(train_path)
all_cols = pf.schema.names
feature_cols = [c for c in all_cols if 'feature' in c]

n_components = 100
batch_size = 50000

ipca = IncrementalPCA(n_components=n_components, batch_size=batch_size)

# 1차: 배치 단위로 읽으면서 partial_fit (전체를 메모리에 안 올림)
for batch in pf.iter_batches(batch_size=batch_size, columns=feature_cols):
    df_batch = batch.to_pandas()
    X_batch = (df_batch.values.astype('float32')) / 4.0  # 0~4 범위를 0~1로 정규화
    ipca.partial_fit(X_batch)
    del df_batch, X_batch

gc.collect()
print("PCA fit 완료")
print(f"설명된 분산 비율 합: {ipca.explained_variance_ratio_.sum():.4f}")

PCA fit 완료
설명된 분산 비율 합: 0.5277


In [3]:
X_train_pca_list = []
for batch in pf.iter_batches(batch_size=batch_size, columns=feature_cols):
    df_batch = batch.to_pandas()
    X_batch = (df_batch.values.astype('float32')) / 4.0
    X_train_pca_list.append(ipca.transform(X_batch))
    del df_batch, X_batch

X_train_pca = np.vstack(X_train_pca_list)
print(X_train_pca.shape)

(2746268, 100)


In [4]:
print(ipca.explained_variance_ratio_.cumsum())

[0.05290579 0.09331523 0.12040723 0.14172254 0.16168043 0.18006856
 0.19696999 0.20963906 0.22198756 0.23283005 0.24353674 0.2531192
 0.26226289 0.27084355 0.27880826 0.2857022  0.29248602 0.29869334
 0.30483008 0.31076871 0.3163788  0.32184371 0.32706299 0.33187238
 0.33654445 0.34117726 0.34560071 0.34991484 0.35421203 0.35831375
 0.36235852 0.36631857 0.37019577 0.37399259 0.37773049 0.38138823
 0.38499424 0.38857247 0.39211433 0.39556406 0.39893204 0.40219913
 0.40543509 0.40863805 0.41176385 0.41487252 0.41794434 0.42100308
 0.42397456 0.42687378 0.42966196 0.43241013 0.43514151 0.43781441
 0.44044258 0.44300167 0.44553437 0.44802948 0.4504906  0.45293552
 0.45534959 0.45770497 0.46004274 0.46236122 0.46466645 0.4669183
 0.46912613 0.47132398 0.47350388 0.47567948 0.47780704 0.47991518
 0.48200512 0.48407515 0.48612534 0.4881632  0.49014632 0.49209742
 0.49400934 0.49590073 0.49777491 0.49961314 0.50140577 0.50317234
 0.50492144 0.50663652 0.50830845 0.50996762 0.51157093 0.513154

In [5]:
np.save('/content/drive/MyDrive/ESAA/X_train_pca.npy', X_train_pca)

In [6]:
import pyarrow.parquet as pq
import numpy as np
import gc

val_path = '/content/drive/MyDrive/ESAA/numerai_r1300__v5_2_validation.parquet'

pf_val = pq.ParquetFile(val_path)

X_val_pca_list = []
for batch in pf_val.iter_batches(batch_size=batch_size, columns=feature_cols):
    df_batch = batch.to_pandas()
    X_batch = (df_batch.values.astype('float32')) / 4.0
    X_val_pca_list.append(ipca.transform(X_batch))
    del df_batch, X_batch

X_val_pca = np.vstack(X_val_pca_list)
del X_val_pca_list
gc.collect()

print("val PCA 완료:", X_val_pca.shape)

val PCA 완료: (4071435, 100)


In [7]:
np.save('/content/drive/MyDrive/ESAA/X_val_pca.npy', X_val_pca)
print("저장 완료")

저장 완료


In [8]:
print(X_train_pca[:5])  # 값이 이상하지 않은지 (NaN, inf 없는지) 육안 확인
print(np.isnan(X_train_pca).sum())  # NaN 개수 체크

[[-2.16408634e+00  2.23581571e+00 -4.78464401e-01  6.67398298e-01
   2.07353249e-01  1.36815601e+00  3.19994646e+00 -9.09335678e-01
  -1.08161074e+00 -5.95903645e-02 -1.87842680e-01  4.92663378e-01
   1.29813231e-01  2.00205842e+00  1.00907714e-01 -3.70210287e-02
  -4.43989198e-01 -2.85230693e-01  8.21114381e-01 -1.06928892e+00
   1.24099962e+00 -8.48707114e-01  1.66099243e+00 -1.37627213e+00
   7.40945461e-01 -4.66034369e-02 -5.12435091e-01  3.39832779e-01
   9.51173767e-01  9.58681550e-01 -8.24318346e-02  5.13422433e-01
   8.62425204e-01 -5.46005405e-01  2.57059201e-01 -1.92596748e-01
  -4.71782208e-01 -1.15989815e-01  1.97693329e-01  1.33818904e+00
  -1.14956748e-01 -3.90094913e-01 -1.23676891e+00  3.14627503e-01
  -1.27535560e+00  1.07247927e+00  5.14350621e-01  2.14369679e-02
   2.19316291e-01  5.50999042e-01 -2.31082362e-01  1.51855111e+00
   5.03590890e-01 -6.10742086e-01  1.17161452e+00 -1.58303869e-01
   4.14009315e-01  3.68042452e-01  5.88007346e-01  2.58299992e-01
  -1.22483